
# PCA-Based EDAD: DDoS Attack Detection
Single-notebook reproduction of the Scientific Reports (2025) paper.

**Datasets (local):**
- `cicids2017.csv`
- `cicids2018.csv`
- `cicddos2019.csv`

Change `DATASET_NAME` below to switch datasets.


In [1]:

# Cell 1: Imports & Global Settings
import numpy as np
import pandas as pd
import random
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)


ModuleNotFoundError: No module named 'imblearn'

In [ ]:

# Cell 2: Dataset Selection & Loading
DATASET_NAME = "cicids2017"  
# options: cicids2017, cicids2018, cicddos2019

DATASET_PATHS = {
    "cicids2017": "cicids2017.csv",
    "cicids2018": "cicids2018.csv",
    "cicddos2019": "cicddos2019.csv"
}

df = pd.read_csv(DATASET_PATHS[DATASET_NAME])
print("Dataset:", DATASET_NAME)
print("Shape:", df.shape)
df.head()


In [ ]:

# Cell 3: Data Cleaning & Preprocessing
df = df.drop_duplicates()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Identify label column (commonly 'Label')
label_col = 'Label' if 'Label' in df.columns else df.columns[-1]

le = LabelEncoder()
df[label_col] = le.fit_transform(df[label_col])

X = df.drop(columns=[label_col])
y = df[label_col]

print("Features after cleaning:", X.shape[1])
print("Class distribution:\n", y.value_counts())


In [ ]:

# Cell 4: Train-Test Split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)


In [ ]:

# Cell 5: Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:

# Cell 6: Handle Class Imbalance using SMOTE (train only)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE class distribution:", np.bincount(y_train_bal))


In [ ]:

# Cell 7: Feature Selection using PCA
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_bal)
X_test_pca = pca.transform(X_test_scaled)

print("PCA components:", X_train_pca.shape[1])


In [ ]:

# Cell 8: Define Models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}


In [ ]:

# Cell 9: Train Models & Predict
results = {}
probas = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train_bal)
    y_pred = model.predict(X_test_pca)
    y_prob = model.predict_proba(X_test_pca)[:, 1] if hasattr(model, "predict_proba") else None

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Confusion": confusion_matrix(y_test, y_pred)
    }
    probas[name] = y_prob

pd.DataFrame(results).T


In [ ]:

# Cell 10: Confusion Matrices
plt.figure(figsize=(15,8))
for i, (name, res) in enumerate(results.items(), 1):
    plt.subplot(2, 3, i)
    sns.heatmap(res["Confusion"], annot=True, fmt='d', cmap='Blues')
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 11: Precision-Recall Curves
plt.figure(figsize=(8,6))
for name, y_prob in probas.items():
    if y_prob is not None:
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        plt.plot(recall, precision, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.grid()
plt.show()


In [ ]:

# Cell 12: Accuracy Comparison
accs = {name: res["Accuracy"] for name, res in results.items()}
plt.figure(figsize=(8,5))
sns.barplot(x=list(accs.keys()), y=list(accs.values()))
plt.xticks(rotation=45)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.ylim(0.8,1.0)
plt.show()
